# Consumer Complaints Analysis

## Project Overview

**Task:** Multi-class text classification — predict which financial product a consumer complaint is about, using only the complaint narrative text.

**Dataset:** [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/) — real complaints filed by US consumers about financial products and services (CC0 public domain).

**Approach:**
1. **Baseline:** TF-IDF vectorization + classical ML (Naive Bayes, Logistic Regression, LinearSVC, Random Forest)
2. **Advanced:** ModernBERT fine-tuning on GPU with mixed-precision training

---

## Learning Objectives

| # | Skill |
|---|-------|
| 1 | Load, clean, and explore a real-world NLP text classification dataset |
| 2 | Understand TF-IDF vectorization for converting text to numerical features |
| 3 | Compare classifiers systematically using cross-validation |
| 4 | Use chi-squared feature analysis to find discriminative terms per category |
| 5 | Interpret confusion matrices and per-class precision/recall/F1 |
| 6 | Analyze misclassification patterns and understand model errors |
| 7 | Fine-tune ModernBERT for text classification on local GPU |
| 8 | Decide when classical baselines are sufficient vs. when transformers help |


## Problem Statement

When consumers file complaints about financial products (mortgages, credit cards, debt collection, etc.), each complaint must be routed to the right department. Manually reading and classifying thousands of complaints is slow and error-prone.

**Goal:** Build a classifier that reads the complaint narrative and predicts the correct product category automatically.

This is a **multi-class text classification** problem with:
- ~10-15 product categories (the exact count depends on dataset filtering)
- Free-form text of varying lengths (50 to 5000+ words)
- Imbalanced classes (some products like "Credit reporting" dominate)

## Why This Project Matters

- **Practical NLP skill:** Text classification is the most common NLP task in industry
- **Real data:** CFPB complaints are noisy, verbose, and messy — like real-world production data
- **Baseline-first mindset:** Shows that TF-IDF + LinearSVC is often surprisingly competitive with transformers
- **Progressive complexity:** You learn when to reach for expensive models vs. when simple baselines work


## Local-First Architecture

```
Complaint Text
      |
      v
  [Preprocessing] -- drop nulls, lowercase, strip noise
      |
      v
  [TF-IDF Vectorization] -- unigrams + bigrams, max 50K features
      |
      +-----> [Baseline Classifiers] -- NB, LogReg, SVM, RF (cross-validated)
      |            |
      |            v
      |       Best classical model (LinearSVC typically wins)
      |
      +-----> [ModernBERT Fine-Tuning] -- GPU, mixed-precision, 2 epochs
                   |
                   v
              [Compare Results]
```

## Stack

| Component | Choice | Why |
|-----------|--------|-----|
| Data source | HuggingFace `datasets` | Clean download, no auth needed |
| Text features | TF-IDF (sklearn) | Fast, interpretable, strong baseline |
| Classical ML | LinearSVC, LogReg, NB, RF | Cross-validated comparison |
| Transformer | ModernBERT (`answerdotai/ModernBERT-base`) | State-of-the-art English encoder, 2024+ |
| GPU training | PyTorch + CUDA + mixed-precision | Fast local training |
| Evaluation | sklearn metrics | Classification report, confusion matrix |

## Model Choice

- **Baseline:** LinearSVC with TF-IDF — fast, interpretable, often 85-95% accurate on text classification
- **Advanced:** [ModernBERT](https://huggingface.co/answerdotai/ModernBERT-base) — modern English-first encoder that outperforms BERT/RoBERTa on many benchmarks while being efficient
- **No cloud APIs required** — everything runs locally on your GPU


## Dataset Overview

The **Consumer Financial Protection Bureau (CFPB)** maintains a public database of complaints about financial products and services. Each complaint includes:

| Column | Description |
|--------|-------------|
| `Product` | Financial product category (target label) |
| `Consumer complaint narrative` | Free-text description written by the consumer |
| `Company` | Company the complaint is about |
| `Issue` / `Sub-issue` | Specific issue category |
| `State` | Consumer's state |
| `Date received` | When the complaint was filed |

**For this project we use only two columns:** the complaint narrative (input text) and the product category (target label).

### Source & License

- **Source:** [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/)
- **HuggingFace:** Available as `CFPB/consumer-finance-complaints`
- **License:** CC0 1.0 (public domain)
- **Size:** Millions of complaints total; we use a filtered subset with narrative text (~100K-200K rows)

### Known Limitations

- Not all complaints have narrative text (consumer must consent to publication)
- Some categories have very similar names (e.g., credit reporting variants)
- Text quality varies: some complaints are detailed, others are a single sentence
- Class imbalance: "Credit reporting" complaints vastly outnumber others


## Environment Setup

The following cell installs required packages. Skip if already installed.


In [ ]:
# Install dependencies (skip if already installed)
# !pip install -q datasets scikit-learn matplotlib seaborn transformers accelerate torch
print("Dependencies: datasets, scikit-learn, matplotlib, seaborn, transformers, torch")
print("(Uncomment the pip install line above if any are missing)")


## Imports


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

print("All imports loaded successfully")


## Configuration

All tuneable parameters in one place. Change the model name, sample sizes, or hyperparameters here.


In [ ]:
# ── Data ──────────────────────────────────────────────────
MAX_ROWS = 200_000        # Max rows to load from the dataset
MIN_CATEGORY_SAMPLES = 500  # Drop categories with fewer samples than this
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ── TF-IDF ────────────────────────────────────────────────
TFIDF_MAX_FEATURES = 50_000
TFIDF_NGRAM_RANGE = (1, 2)   # Unigrams + bigrams
TFIDF_MIN_DF = 5

# ── Cross-Validation ─────────────────────────────────────
CV_FOLDS = 5

# ── ModernBERT (Advanced Section) ─────────────────────────
BERT_MODEL_NAME = "answerdotai/ModernBERT-base"
BERT_MAX_SAMPLES = 15_000   # Subset for transformer fine-tuning
BERT_MAX_LEN = 256
BERT_BATCH_SIZE = 16
BERT_EPOCHS = 2
BERT_LR = 2e-5

print("Configuration loaded")


## Dataset Loading

We load the CFPB Consumer Complaints dataset from HuggingFace. The official dataset contains millions of rows, but many lack complaint narratives. We load a manageable subset and filter to rows that have narrative text.


In [ ]:
from datasets import load_dataset

# ── Adaptive data loading with fallback ──────────────────
SOURCES = [
    ("CFPB/consumer-finance-complaints", "Official CFPB dataset (CC0)"),
    ("milesbutler/consumer_complaints", "Pre-processed CFPB mirror (MIT)"),
]

df_raw = None
for ds_name, desc in SOURCES:
    try:
        print(f"Trying: {ds_name} ({desc})...")
        ds = load_dataset(ds_name, split="train", trust_remote_code=True)
        df_raw = ds.to_pandas()
        if len(df_raw) > MAX_ROWS:
            df_raw = df_raw.sample(MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f"Loaded {len(df_raw):,} rows from {ds_name}")
        break
    except Exception as e:
        print(f"  Not available: {e}")

if df_raw is None:
    raise RuntimeError(
        "Could not load from any HuggingFace source.\n"
        "Manual download: https://www.consumerfinance.gov/data-research/consumer-complaints/\n"
        "Save as Consumer_Complaints.csv in this folder, then load with pd.read_csv()."
    )

print(f"\nDataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")


### Column Detection

Different dataset versions may use different column names. We detect the text and target columns automatically.


In [ ]:
# ── Detect text column ────────────────────────────────────
TEXT_CANDIDATES = [
    "Consumer complaint narrative", "consumer_complaint_narrative",
    "consumer_narrative", "narrative", "complaint_text", "text",
]
text_col = next((c for c in TEXT_CANDIDATES if c in df_raw.columns), None)
if text_col is None:
    obj_cols = df_raw.select_dtypes("object").columns
    text_col = max(obj_cols, key=lambda c: df_raw[c].astype(str).str.len().mean())
    print(f"Auto-detected text column: '{text_col}'")

# ── Detect target column ─────────────────────────────────
TARGET_CANDIDATES = ["Product", "product", "category", "label"]
target_col = next((c for c in TARGET_CANDIDATES if c in df_raw.columns), None)
if target_col is None:
    raise ValueError(f"No target column found. Available: {list(df_raw.columns)}")

print(f"Text column:   '{text_col}'")
print(f"Target column: '{target_col}'")

# ── Standardize to consistent names ──────────────────────
df = df_raw[[text_col, target_col]].copy()
df.columns = ["text", "product"]

# ── Filter: keep rows with actual narrative text ─────────
before = len(df)
df = df.dropna(subset=["text"])
df = df[df["text"].astype(str).str.strip().str.len() > 20]
df = df.reset_index(drop=True)
print(f"\nFiltered: {before:,} -> {len(df):,} rows (removed {before - len(df):,} without text)")


## Data Validation

Defensive checks before we proceed: verify data quality, check for missing values, duplicates, and class distribution.


In [ ]:
# ── Basic quality checks ──────────────────────────────────
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate texts: {df['text'].duplicated().sum():,}")

# ── Category distribution ────────────────────────────────
print(f"\nUnique product categories: {df['product'].nunique()}")
print("\nCategory counts:")
for cat, count in df["product"].value_counts().items():
    print(f"  {cat}: {count:,}")

# ── Filter small categories ──────────────────────────────
cat_counts = df["product"].value_counts()
valid_cats = cat_counts[cat_counts >= MIN_CATEGORY_SAMPLES].index
removed = df["product"].nunique() - len(valid_cats)
df = df[df["product"].isin(valid_cats)].reset_index(drop=True)
print(f"\nKept {len(valid_cats)} categories (removed {removed} with < {MIN_CATEGORY_SAMPLES} samples)")
print(f"Final dataset: {len(df):,} rows, {df['product'].nunique()} categories")


## Exploratory Data Analysis

### Class Distribution

Class imbalance is common in complaint data — some product categories receive far more complaints than others. Understanding this helps interpret model performance later.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
order = df["product"].value_counts().index
sns.countplot(data=df, y="product", order=order, ax=ax, color="steelblue")
ax.set_title("Complaint Count by Product Category", fontsize=14)
ax.set_xlabel("Number of Complaints")
ax.set_ylabel("")

# Add percentage labels
total = len(df)
for i, (count, cat) in enumerate(zip(df["product"].value_counts(), order)):
    ax.text(count + total * 0.005, i, f"{count:,} ({count/total:.1%})", va="center", fontsize=9)

plt.tight_layout()
plt.show()


### Text Length Analysis

Understanding text length distribution helps choose model parameters (TF-IDF max features, transformer max sequence length) and anticipate truncation effects.


In [ ]:
df["text_len"] = df["text"].astype(str).str.len()
df["word_count"] = df["text"].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length distribution
axes[0].hist(df["text_len"].clip(upper=df["text_len"].quantile(0.99)),
             bins=60, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("Text Length Distribution (characters)")
axes[0].set_xlabel("Characters")
axes[0].axvline(df["text_len"].median(), color="red", linestyle="--", label=f"Median: {df['text_len'].median():.0f}")
axes[0].legend()

# Word count distribution
axes[1].hist(df["word_count"].clip(upper=df["word_count"].quantile(0.99)),
             bins=60, color="darkorange", edgecolor="white", alpha=0.8)
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Words")
axes[1].axvline(df["word_count"].median(), color="red", linestyle="--", label=f"Median: {df['word_count'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats
print(f"Text length — mean: {df['text_len'].mean():.0f}, median: {df['text_len'].median():.0f}, "
      f"min: {df['text_len'].min()}, max: {df['text_len'].max()}")
print(f"Word count  — mean: {df['word_count'].mean():.0f}, median: {df['word_count'].median():.0f}, "
      f"min: {df['word_count'].min()}, max: {df['word_count'].max()}")

# Clean up temporary columns
df = df.drop(columns=["text_len", "word_count"])


## Preprocessing

For TF-IDF, minimal preprocessing is usually best — the vectorizer handles lowercasing and stop word removal. We just ensure consistent formatting.

> **Why minimal preprocessing?** TF-IDF with stop word removal already handles most noise. Over-cleaning (stemming, lemmatizing) can actually hurt performance by destroying useful signal.


In [ ]:
# ── Encode categories ─────────────────────────────────────
df["category_id"] = df["product"].factorize()[0]

category_map = (
    df[["product", "category_id"]]
    .drop_duplicates()
    .sort_values("category_id")
    .reset_index(drop=True)
)
category_to_id = dict(zip(category_map["product"], category_map["category_id"]))
id_to_category = dict(zip(category_map["category_id"], category_map["product"]))

print("Category mapping:")
for cat, cid in sorted(category_to_id.items(), key=lambda x: x[1]):
    count = (df["category_id"] == cid).sum()
    print(f"  [{cid:2d}] {cat} ({count:,} samples)")


## TF-IDF Feature Engineering

**TF-IDF** (Term Frequency — Inverse Document Frequency) converts text to numerical vectors where:
- **TF:** How often a term appears in a document (more = more important for that doc)
- **IDF:** How rare the term is across all documents (rarer = more discriminative)

We use unigrams + bigrams to capture both single words and two-word phrases.


In [ ]:
tfidf = TfidfVectorizer(
    sublinear_tf=True,          # Use log(1 + tf) instead of raw counts
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,        # Ignore terms appearing in < 5 documents
    norm="l2",                  # L2 normalize each row
    encoding="latin-1",
    ngram_range=TFIDF_NGRAM_RANGE,
    stop_words="english",
)

features = tfidf.fit_transform(df["text"])
labels = df["category_id"].values

print(f"TF-IDF matrix shape: {features.shape}")
print(f"  {features.shape[0]:,} documents x {features.shape[1]:,} features")
print(f"  Sparsity: {1 - features.nnz / (features.shape[0] * features.shape[1]):.4%}")


## Chi-Squared Feature Analysis

Chi-squared ($\chi^2$) test measures how strongly each feature (word/bigram) is associated with each category. High $\chi^2$ = the feature is very discriminative for that category.

This helps us understand **what the model is actually learning** — which words distinguish one product category from another.


In [ ]:
N_TOP = 3  # Top N features to show per category
feature_names = np.array(tfidf.get_feature_names_out())

for product, cat_id in sorted(category_to_id.items(), key=lambda x: x[1]):
    chi2_scores, _ = chi2(features, labels == cat_id)
    top_idx = np.argsort(chi2_scores)[-20:]  # Top 20 by chi2

    top_features = feature_names[top_idx]
    unigrams = [f for f in reversed(top_features) if len(f.split()) == 1][:N_TOP]
    bigrams  = [f for f in reversed(top_features) if len(f.split()) == 2][:N_TOP]

    print(f"# '{product}':")
    print(f"  Top unigrams: {', '.join(unigrams)}")
    print(f"  Top bigrams:  {', '.join(bigrams)}")
    print()


## Baseline Models — Cross-Validation Comparison

We compare four classical classifiers using 5-fold cross-validation on TF-IDF features:

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **MultinomialNB** | Very fast, good baseline for text | Assumes feature independence |
| **LogisticRegression** | Strong, interpretable, handles imbalance | Slower on high-dim data |
| **LinearSVC** | Best for high-dimensional sparse data like TF-IDF | No probability outputs by default |
| **RandomForest** | Handles non-linear patterns | Poor on sparse high-dimensional text |

> **Why cross-validation?** A single train/test split can be misleading — CV gives us a more robust estimate of model performance with confidence intervals.


In [ ]:
models = [
    ("MultinomialNB",     MultinomialNB()),
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)),
    ("LinearSVC",          LinearSVC(max_iter=2000, random_state=RANDOM_STATE)),
    ("RandomForest",       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)),
]

cv_results = []
for name, model in models:
    print(f"  Cross-validating {name}...", end=" ")
    scores = cross_val_score(model, features, labels, cv=CV_FOLDS, scoring="accuracy", n_jobs=-1)
    for fold, score in enumerate(scores):
        cv_results.append({"model": name, "fold": fold, "accuracy": score})
    print(f"mean={scores.mean():.4f} +/- {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results)
print("\nCross-validation complete!")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=cv_df, x="model", y="accuracy", ax=ax, palette="Set2")
sns.stripplot(data=cv_df, x="model", y="accuracy", ax=ax,
              size=8, jitter=True, edgecolor="gray", linewidth=1.5, alpha=0.7)
ax.set_title(f"Model Comparison ({CV_FOLDS}-Fold Cross-Validation)", fontsize=14)
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Summary table
summary = cv_df.groupby("model")["accuracy"].agg(["mean", "std"]).sort_values("mean", ascending=False)
summary.columns = ["Mean Accuracy", "Std"]
print(summary.to_string())


## Best Classical Model — Training & Evaluation

Based on cross-validation, **LinearSVC** typically achieves the best accuracy on TF-IDF text features. We now train it on the full training set and evaluate on a held-out test set.


In [ ]:
# ── Train/test split ──────────────────────────────────────
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    features, labels, df.index,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=labels,
)
print(f"Train: {X_train.shape[0]:,} samples")
print(f"Test:  {X_test.shape[0]:,} samples")

# ── Train LinearSVC ──────────────────────────────────────
best_model = LinearSVC(max_iter=2000, random_state=RANDOM_STATE)
best_model.fit(X_train, y_train)

# ── Predict ──────────────────────────────────────────────
y_pred = best_model.predict(X_test)

# ── Classification Report ────────────────────────────────
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")
print(f"\nLinearSVC Results:")
print(f"  Accuracy:     {acc:.4f}")
print(f"  F1 (weighted): {f1:.4f}")
print()
print(classification_report(
    y_test, y_pred,
    target_names=[id_to_category[i] for i in sorted(id_to_category)],
    zero_division=0,
))


### Confusion Matrix

The confusion matrix shows which categories get confused with each other. Dark diagonal = good predictions. Off-diagonal = misclassifications.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cat_names = [id_to_category[i] for i in sorted(id_to_category)]

# Shorten long category names for display
short_names = []
for name in cat_names:
    if len(name) > 30:
        short_names.append(name[:28] + "..")
    else:
        short_names.append(name)

fig, ax = plt.subplots(figsize=(max(10, len(cat_names) * 0.9), max(8, len(cat_names) * 0.8)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=short_names, yticklabels=short_names)
ax.set_ylabel("Actual", fontsize=12)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_title("LinearSVC Confusion Matrix", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()


## Error Analysis

Understanding **why** the model fails is more valuable than the accuracy number. We look at the most common misclassification pairs and inspect actual examples.

This reveals:
- Categories with ambiguous overlap (e.g., "Credit card" vs "Bank account")
- Edge cases where the complaint text is genuinely confusing
- Potential labeling issues in the dataset


In [ ]:
# ── Find most confused category pairs ─────────────────────
print("Most Common Misclassifications:\n")
confusion_pairs = []
for actual_id in range(len(cat_names)):
    for pred_id in range(len(cat_names)):
        if actual_id != pred_id and cm[actual_id, pred_id] >= 5:
            confusion_pairs.append((
                id_to_category[actual_id],
                id_to_category[pred_id],
                cm[actual_id, pred_id],
            ))

confusion_pairs.sort(key=lambda x: x[2], reverse=True)
for actual, predicted, count in confusion_pairs[:10]:
    print(f"  '{actual}' misclassified as '{predicted}': {count} examples")

# ── Show example misclassifications ──────────────────────
if confusion_pairs:
    top_actual, top_pred, _ = confusion_pairs[0]
    actual_id = category_to_id[top_actual]
    pred_id = category_to_id[top_pred]

    mask = (y_test == actual_id) & (y_pred == pred_id)
    misclassified_idx = idx_test[mask]

    print(f"\n--- Example: '{top_actual}' predicted as '{top_pred}' ---")
    for i, idx in enumerate(misclassified_idx[:3]):
        text = df.loc[idx, "text"]
        preview = text[:300] + "..." if len(text) > 300 else text
        print(f"\n[{i+1}] {preview}")


## Inference — Predict New Complaints

Test the trained model with new complaint texts. This simulates how the model would work in a production routing system.


In [ ]:
# ── Predict on new (unseen) complaint texts ──────────────
sample_complaints = [
    "This company refuses to provide me verification and validation of debt per my right under the FDCPA. I do not believe this debt is mine.",
    "I requested a home loan modification through my bank but they never responded despite multiple follow-ups over three months.",
    "My credit report shows accounts that do not belong to me. I filed a dispute but the bureau did not investigate properly.",
    "I have been charged overdraft fees multiple times for transactions that should not have triggered them.",
    "I am unable to make my student loan payments and the servicer has not offered me any alternative repayment plans.",
]

# Re-train on all data for best inference model
best_model.fit(features, labels)
new_features = tfidf.transform(sample_complaints)
predictions = best_model.predict(new_features)

print("Predictions on new complaints:\n")
for text, pred_id in zip(sample_complaints, predictions):
    pred_label = id_to_category[pred_id]
    preview = text[:120] + "..." if len(text) > 120 else text
    print(f'  "{preview}"')
    print(f"  -> Predicted: {pred_label}")
    print()


### Model Interpretability — Top Features per Category

After training on all data, we can inspect which words/bigrams have the highest weight for each category in the LinearSVC model. This tells us what the model considers most indicative.


In [ ]:
N_TOP_FEATS = 3

for product, cat_id in sorted(category_to_id.items(), key=lambda x: x[1]):
    top_idx = np.argsort(best_model.coef_[cat_id])
    top_features = feature_names[top_idx]

    unigrams = [f for f in reversed(top_features) if len(f.split()) == 1][:N_TOP_FEATS]
    bigrams  = [f for f in reversed(top_features) if len(f.split()) == 2][:N_TOP_FEATS]

    print(f"# '{product}':")
    print(f"  Top unigrams: {', '.join(unigrams)}")
    print(f"  Top bigrams:  {', '.join(bigrams)}")


## Advanced: ModernBERT Fine-Tuning (GPU)

> **This section requires a CUDA-capable GPU.** It fine-tunes a transformer encoder for sequence classification — significantly more expensive than TF-IDF but often more accurate on complex text.

**ModernBERT** (`answerdotai/ModernBERT-base`) is a modern English-focused encoder that improves on BERT and RoBERTa with better training recipes and architecture tweaks.

We use:
- Mixed-precision (fp16) for faster training
- Learning rate warmup + linear decay
- Gradient clipping for stability
- A smaller data subset to keep training time reasonable (~5-15 min on RTX 4060/4070)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# ── Prepare data ─────────────────────────────────────────
bert_df = df[["text", "product"]].copy()
if len(bert_df) > BERT_MAX_SAMPLES:
    bert_df = bert_df.sample(BERT_MAX_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

le = LabelEncoder()
bert_df["label_id"] = le.fit_transform(bert_df["product"])
n_classes = len(le.classes_)

bert_train, bert_test = train_test_split(
    bert_df, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=bert_df["label_id"],
)
print(f"\nModernBERT data - Train: {len(bert_train):,}, Test: {len(bert_test):,}, Classes: {n_classes}")

# ── Tokenizer + Model ────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME, num_labels=n_classes
).to(device)

class ComplaintDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=BERT_MAX_LEN, return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.enc.items()}, "labels": self.labels[i]}

train_loader = DataLoader(
    ComplaintDataset(bert_train["text"].tolist(), bert_train["label_id"].tolist()),
    batch_size=BERT_BATCH_SIZE, shuffle=True,
)
test_loader = DataLoader(
    ComplaintDataset(bert_test["text"].tolist(), bert_test["label_id"].tolist()),
    batch_size=BERT_BATCH_SIZE,
)

# ── Training ─────────────────────────────────────────────
use_amp = device.type == "cuda"
optimizer = torch.optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=0.01)
total_steps = len(train_loader) * BERT_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

t0 = time.perf_counter()
for epoch in range(BERT_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    elapsed = time.perf_counter() - t0
    print(f"  Epoch {epoch+1}/{BERT_EPOCHS} — Loss: {avg_loss:.4f} ({elapsed:.0f}s elapsed)")

total_time = time.perf_counter() - t0
print(f"\nTraining complete in {total_time:.0f}s")


In [ ]:
# ── Evaluation ────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

bert_acc = accuracy_score(all_labels, all_preds)
bert_f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"ModernBERT Results:")
print(f"  Accuracy:      {bert_acc:.4f}")
print(f"  F1 (weighted): {bert_f1:.4f}")
print()
print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    zero_division=0,
))

# ── Confusion Matrix ─────────────────────────────────────
bert_cm = confusion_matrix(all_labels, all_preds)
bert_names = list(le.classes_)
bert_short = [n[:28] + ".." if len(n) > 30 else n for n in bert_names]

fig, ax = plt.subplots(figsize=(max(10, len(bert_names) * 0.9), max(8, len(bert_names) * 0.8)))
sns.heatmap(bert_cm, annot=True, fmt="d", cmap="Greens", ax=ax,
            xticklabels=bert_short, yticklabels=bert_short)
ax.set_ylabel("Actual", fontsize=12)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_title("ModernBERT Confusion Matrix", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()


## Model Comparison

Side-by-side comparison of all models — classical baselines and ModernBERT. This summarizes the accuracy–speed tradeoff.


In [ ]:
results = []

# Add cross-validation results
for model_name in cv_df["model"].unique():
    mask = cv_df["model"] == model_name
    mean_acc = cv_df.loc[mask, "accuracy"].mean()
    results.append({"Model": model_name, "Accuracy": round(mean_acc, 4), "Type": "TF-IDF + Classical", "Note": f"{CV_FOLDS}-fold CV mean"})

# Add LinearSVC test set result
results.append({"Model": "LinearSVC (test set)", "Accuracy": round(acc, 4), "Type": "TF-IDF + Classical", "Note": "Held-out test set"})

# Add ModernBERT (if it ran)
try:
    results.append({"Model": "ModernBERT", "Accuracy": round(bert_acc, 4), "Type": "Transformer", "Note": f"{BERT_MAX_SAMPLES:,} samples, {BERT_EPOCHS} epochs"})
except NameError:
    print("(ModernBERT section was not run)")

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))


## Evaluation Summary

### What the Metrics Mean

| Metric | Meaning | Good Value |
|--------|---------|------------|
| **Accuracy** | % of complaints classified correctly overall | > 80% |
| **F1 (weighted)** | Harmonic mean of precision and recall, weighted by class size | > 80% |
| **Precision** (per-class) | Of all complaints predicted as X, how many actually are X? | Higher = fewer false positives |
| **Recall** (per-class) | Of all actual X complaints, how many did we find? | Higher = fewer missed complaints |

### Key Observations

- **LinearSVC** with TF-IDF is typically very competitive (85-95% accuracy) and trains in seconds
- **ModernBERT** may add 1-3% accuracy improvement but takes minutes to train and requires GPU
- **Misclassifications** often involve genuinely ambiguous categories (e.g., "Credit card" vs "Bank account" when the complaint mentions both)
- **Class imbalance** affects rare categories — they may have lower recall


## Limitations

1. **Single-label only:** Each complaint gets one product label, but some genuinely span multiple products
2. **English only:** The dataset is US-focused; model won't generalize to other languages
3. **Category drift:** CFPB has changed product category names over time; older complaints may use different labels
4. **No temporal split:** We use random splits, not time-based splits — this may overestimate performance
5. **Truncation:** ModernBERT truncates to 256 tokens — very long complaints lose information
6. **No calibration:** Model confidence scores are unreliable without calibration
7. **Sensitive data:** Although CFPB redacts PII (XXXX), some complaints may still contain identifying patterns


## How to Improve This Project

| Improvement | Difficulty | Expected Impact |
|-------------|------------|-----------------|
| **Temporal train/test split** (train on older, test on newer) | Easy | More realistic performance estimate |
| **Category consolidation** (merge similar product categories) | Easy | Higher accuracy, simpler model |
| **Class-weighted training** (up-weight rare categories) | Easy | Better recall on minority classes |
| **Longer context** (increase ModernBERT max_len to 512) | Easy | Better for long complaints |
| **Ensemble** (combine TF-IDF + transformer predictions) | Medium | Usually +1-2% accuracy |
| **Hyperparameter tuning** (GridSearchCV on TF-IDF params) | Medium | Modest gains |
| **Data augmentation** (paraphrase rare-class complaints) | Medium | Better minority class performance |
| **Hierarchical classification** (predict product, then sub-product) | Hard | Finer-grained routing |
| **Multi-label classification** (allow multiple products) | Hard | Handles ambiguous complaints |

## Production Considerations

- **Latency:** TF-IDF + LinearSVC predicts in <1ms per complaint. ModernBERT takes ~10-50ms on GPU.
- **Model size:** TF-IDF vectorizer + SVC model is ~100-500MB. ModernBERT is ~500MB.
- **Monitoring:** Track prediction distribution over time to detect category drift.
- **Confidence threshold:** Reject predictions with low confidence and route to human review.
- **A/B testing:** Compare model predictions vs human labels on a sample before full deployment.
- **Re-training cadence:** Retrain quarterly as new complaint patterns emerge.


## Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| Using accuracy alone on imbalanced data | Predicting the majority class gives high accuracy but useless model | Use per-class F1 and confusion matrix |
| Not removing rows without narrative text | Many CFPB rows have no narrative — model gets garbage input | Filter to non-null narratives before training |
| Over-processing text (heavy stemming, regex cleaning) | Destroys useful signal; TF-IDF handles most noise internally | Keep preprocessing minimal |
| Evaluating on training data | Gives inflated metrics that don't reflect real performance | Always use held-out test set or cross-validation |
| Fine-tuning a transformer when TF-IDF works fine | Wasted GPU time, higher latency, no meaningful accuracy gain | Start with baselines; only upgrade if needed |
| Ignoring class imbalance | Rare categories get poor recall | Use class weights or stratified sampling |
| Using random split for time-series data | Leaks future information into training | Use temporal split in production |


## Mini Challenge

Test your understanding with these exercises:

1. **Change the TF-IDF parameters:** Set `ngram_range=(1, 3)` (include trigrams) and `max_features=100000`. Does accuracy improve? Why or why not?

2. **Add class weights to LinearSVC:** Use `LinearSVC(class_weight="balanced")`. How does this change recall for rare categories?

3. **Compare different text preprocessing:** Try lowercasing only vs. removing punctuation vs. removing digits. Which produces the best TF-IDF features?

4. **Increase ModernBERT epochs to 4:** Does accuracy keep improving, or does it plateau/overfit? Plot train loss per epoch.

5. **Build a simple ensemble:** Average the LinearSVC prediction (via `decision_function`) with ModernBERT logits. Does the ensemble beat both individual models?

6. **Error deep-dive:** Pick the two most confused categories and read 10 misclassified examples. Can you identify a pattern? Could better labeling fix it?


## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **TF-IDF + LinearSVC is a remarkably strong baseline** for text classification — often 85-95% accurate |
| 2 | **Start simple, baseline first.** Only reach for transformers after measuring the baseline gap |
| 3 | **Chi-squared analysis** reveals which words actually drive classification — essential for trust and debugging |
| 4 | **Cross-validation** is more reliable than a single train/test split for comparing models |
| 5 | **Confusion matrix > accuracy** for understanding where your model fails |
| 6 | **Error analysis** on actual misclassified examples teaches you more than any metric |
| 7 | **ModernBERT** can improve over TF-IDF, but the cost (time, GPU, complexity) must justify the gain |
| 8 | **Data quality matters more than model choice** — filtering, cleaning, and category design drive results |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*
